In [ ]:
import os 
os.environ["LANGEXTRACT_API_KEY"] = ""


In [2]:
# ============================================================================
# CELL 1: CONFIGURATION
# ============================================================================

import os
import re
from pathlib import Path

# Required parameters
POLICY_PDF = "UHC_Commercial_Medical_Policy_Adalimumab.pdf"
DISEASE_NAME = "Rheumatoid Arthritis"
DRUG_NAME = "Adalimumab"

# Authorization type to extract
AUTH_TYPE = "Initial Auth"

# Output paths (auto-generated from disease name slug by default)
OUTPUT_DIR = "."


def _slug(name):
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")


disease_slug = _slug(DISEASE_NAME)
auth_slug = _slug(AUTH_TYPE)
OUTPUT_CLEAN_TEXT = f"{disease_slug}_{auth_slug}_clean.txt"
OUTPUT_FLAT_JSONL = f"{disease_slug}_{auth_slug}_extractions.jsonl"
OUTPUT_TREE_JSON = f"{disease_slug}_{auth_slug}_decision_tree.json"
OUTPUT_HTML = f"{disease_slug}_{auth_slug}_visualization.html"

# Model configuration
CLEANING_MODEL = "gemini-2.0-flash"
EXTRACTION_MODEL = "gemini-2.5-flash"
MAX_PDF_PAGES = 9999

# Ensure API key is set from environment
if "LANGEXTRACT_API_KEY" not in os.environ:
    raise ValueError("LANGEXTRACT_API_KEY environment variable not set")

print("Configuration loaded:")
print(f"  Disease: {DISEASE_NAME}")
print(f"  Drug: {DRUG_NAME}")
print(f"  Auth Type: {AUTH_TYPE}")
print(f"  Output files: {disease_slug}_{auth_slug}_*")

Configuration loaded:
  Disease: Rheumatoid Arthritis
  Drug: Adalimumab
  Auth Type: Initial Auth
  Output files: rheumatoid_arthritis_initial_auth_*


In [3]:
# ============================================================================
# CELL 2: IMPORTS + UTILITY FUNCTIONS
# ============================================================================

import importlib
import time
import fitz
import langextract as lx
import google.generativeai as genai
import json
import textwrap
import logging

# Suppress LangExtract's fuzzy-match alignment warnings (MATCH_FUZZY is fine)
logging.getLogger("absl").setLevel(logging.ERROR)

import policy_tree
importlib.reload(policy_tree)
from policy_tree import enrich_tree

# Configure API from environment
genai.configure(api_key=os.environ["LANGEXTRACT_API_KEY"])

# ============================================================================
# GENERIC EXAMPLE FOR LANGEXTRACT (policy-agnostic)
# ============================================================================

# Synthetic policy text that covers all structural patterns without biasing
# toward any specific disease, drug, or payer format.
GENERIC_EXAMPLE_TEXT = (
    "Initial Authorization requires all of the following criteria: "
    "(1) Diagnosis of moderate to severe [condition]. "
    "(2) One of the following: "
    "(a) History of an inadequate response to a 3-month trial of one conventional therapy "
    "[e.g., Drug A, Drug B, Drug C] at maximally tolerated doses "
    "(document drug, date, and duration of trial); "
    "(b) Patient has been previously treated with a targeted therapy "
    "[e.g., Medication X, Medication Y, Medication Z] "
    "as documented by claims history or submission of medical records "
    "(document drug, date, and duration of therapy); "
    "(c) Both of the following: "
    "i. Patient is currently receiving the requested medication "
    "as documented by claims history or submission of medical records "
    "(document date and duration of therapy); "
    "ii. Patient has not received a manufacturer-supplied sample at no cost. "
    "(3) Patient is not receiving the requested medication in combination with "
    "another targeted therapy [e.g., Medication X, Medication Y, Medication Z]. "
    "(4) Prescribed by or in consultation with a relevant specialist."
)

_EXAMPLE_AUTH = "Initial Authorization"

GENERIC_EXAMPLE_EXTRACTIONS = [
    # Root-level logic gate
    lx.data.Extraction(
        extraction_class="LogicGate",
        extraction_text="all of the following criteria",
        attributes={
            "logic_path": json.dumps([f"{_EXAMPLE_AUTH} (AND)"]),
            "type": "LogicGate"
        }
    ),
    # Diagnosis criterion
    lx.data.Extraction(
        extraction_class="Criterion",
        extraction_text="(1) Diagnosis of moderate to severe [condition]",
        attributes={
            "logic_path": json.dumps([f"{_EXAMPLE_AUTH} (AND)", "Diagnosis"]),
            "type": "Mandatory"
        }
    ),
    # Step Therapy logic gate (OR)
    lx.data.Extraction(
        extraction_class="LogicGate",
        extraction_text="(2) One of the following:",
        attributes={
            "logic_path": json.dumps([f"{_EXAMPLE_AUTH} (AND)", "Step Therapy (OR)"]),
            "type": "LogicGate"
        }
    ),
    # Conventional Therapy Failure sub-gate (OR)
    lx.data.Extraction(
        extraction_class="LogicGate",
        extraction_text="(a) History of an inadequate response to a 3-month trial of one conventional therapy",
        attributes={
            "logic_path": json.dumps([f"{_EXAMPLE_AUTH} (AND)", "Step Therapy (OR)", "Conventional Therapy Failure (OR)"]),
            "type": "LogicGate"
        }
    ),
    # Evidence requirement for conventional therapy failure
    lx.data.Extraction(
        extraction_class="EvidenceRequirement",
        extraction_text="document drug, date, and duration of trial",
        attributes={
            "logic_path": json.dumps([f"{_EXAMPLE_AUTH} (AND)", "Step Therapy (OR)", "Conventional Therapy Failure (OR)"]),
            "type": "EvidenceRequirement",
        }
    ),
    # Conventional therapy drugs (3 generic)
    lx.data.Extraction(
        extraction_class="Criterion",
        extraction_text="Drug A",
        attributes={
            "logic_path": json.dumps([f"{_EXAMPLE_AUTH} (AND)", "Step Therapy (OR)", "Conventional Therapy Failure (OR)"]),
            "type": "Drug"
        }
    ),
    lx.data.Extraction(
        extraction_class="Criterion",
        extraction_text="Drug B",
        attributes={
            "logic_path": json.dumps([f"{_EXAMPLE_AUTH} (AND)", "Step Therapy (OR)", "Conventional Therapy Failure (OR)"]),
            "type": "Drug"
        }
    ),
    lx.data.Extraction(
        extraction_class="Criterion",
        extraction_text="Drug C",
        attributes={
            "logic_path": json.dumps([f"{_EXAMPLE_AUTH} (AND)", "Step Therapy (OR)", "Conventional Therapy Failure (OR)"]),
            "type": "Drug"
        }
    ),
    # Prior Targeted Therapy sub-gate (OR)
    lx.data.Extraction(
        extraction_class="LogicGate",
        extraction_text="(b) Patient has been previously treated with a targeted therapy",
        attributes={
            "logic_path": json.dumps([f"{_EXAMPLE_AUTH} (AND)", "Step Therapy (OR)", "Prior Targeted Therapy (OR)"]),
            "type": "LogicGate"
        }
    ),
    # Evidence requirement for prior targeted therapy
    lx.data.Extraction(
        extraction_class="EvidenceRequirement",
        extraction_text="as documented by claims history or submission of medical records (document drug, date, and duration of therapy)",
        attributes={
            "logic_path": json.dumps([f"{_EXAMPLE_AUTH} (AND)", "Step Therapy (OR)", "Prior Targeted Therapy (OR)"]),
            "type": "EvidenceRequirement",
        }
    ),
    # Prior targeted therapy medications (3 generic)
    lx.data.Extraction(
        extraction_class="Criterion",
        extraction_text="Medication X",
        attributes={
            "logic_path": json.dumps([f"{_EXAMPLE_AUTH} (AND)", "Step Therapy (OR)", "Prior Targeted Therapy (OR)"]),
            "type": "Drug"
        }
    ),
    lx.data.Extraction(
        extraction_class="Criterion",
        extraction_text="Medication Y",
        attributes={
            "logic_path": json.dumps([f"{_EXAMPLE_AUTH} (AND)", "Step Therapy (OR)", "Prior Targeted Therapy (OR)"]),
            "type": "Drug"
        }
    ),
    lx.data.Extraction(
        extraction_class="Criterion",
        extraction_text="Medication Z",
        attributes={
            "logic_path": json.dumps([f"{_EXAMPLE_AUTH} (AND)", "Step Therapy (OR)", "Prior Targeted Therapy (OR)"]),
            "type": "Drug"
        }
    ),
    # Current User logic gate (AND — "both of the following")
    lx.data.Extraction(
        extraction_class="LogicGate",
        extraction_text="(c) Both of the following:",
        attributes={
            "logic_path": json.dumps([f"{_EXAMPLE_AUTH} (AND)", "Step Therapy (OR)", "Current User (AND)"]),
            "type": "LogicGate"
        }
    ),
    # Evidence requirement targeting a specific sub-criterion
    lx.data.Extraction(
        extraction_class="EvidenceRequirement",
        extraction_text="as documented by claims history or submission of medical records (document date and duration of therapy)",
        attributes={
            "logic_path": json.dumps([f"{_EXAMPLE_AUTH} (AND)", "Step Therapy (OR)", "Current User (AND)", "i. Patient is currently receiving the requested medication"]),
            "type": "EvidenceRequirement",
        }
    ),
    # Current user sub-requirements
    lx.data.Extraction(
        extraction_class="Criterion",
        extraction_text="i. Patient is currently receiving the requested medication as documented by claims history or submission of medical records",
        attributes={
            "logic_path": json.dumps([f"{_EXAMPLE_AUTH} (AND)", "Step Therapy (OR)", "Current User (AND)"]),
            "type": "SubRequirement"
        }
    ),
    lx.data.Extraction(
        extraction_class="Criterion",
        extraction_text="ii. Patient has not received a manufacturer-supplied sample at no cost",
        attributes={
            "logic_path": json.dumps([f"{_EXAMPLE_AUTH} (AND)", "Step Therapy (OR)", "Current User (AND)"]),
            "type": "SubRequirement"
        }
    ),
    # Combination therapy criterion (negated)
    lx.data.Extraction(
        extraction_class="Criterion",
        extraction_text="(3) Patient is not receiving the requested medication in combination with another targeted therapy",
        attributes={
            "logic_path": json.dumps([f"{_EXAMPLE_AUTH} (AND)", "Combination Therapy"]),
            "type": "Mandatory",
            "negated": "true"
        }
    ),
    # Combination therapy drug list
    lx.data.Extraction(
        extraction_class="Criterion",
        extraction_text="Medication X",
        attributes={
            "logic_path": json.dumps([f"{_EXAMPLE_AUTH} (AND)", "Combination Therapy (OR)"]),
            "type": "Drug"
        }
    ),
    lx.data.Extraction(
        extraction_class="Criterion",
        extraction_text="Medication Y",
        attributes={
            "logic_path": json.dumps([f"{_EXAMPLE_AUTH} (AND)", "Combination Therapy (OR)"]),
            "type": "Drug"
        }
    ),
    lx.data.Extraction(
        extraction_class="Criterion",
        extraction_text="Medication Z",
        attributes={
            "logic_path": json.dumps([f"{_EXAMPLE_AUTH} (AND)", "Combination Therapy (OR)"]),
            "type": "Drug"
        }
    ),
    # Prescriber criterion
    lx.data.Extraction(
        extraction_class="Criterion",
        extraction_text="(4) Prescribed by or in consultation with a relevant specialist",
        attributes={
            "logic_path": json.dumps([f"{_EXAMPLE_AUTH} (AND)", "Prescriber"]),
            "type": "Mandatory"
        }
    ),
]

# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def read_pdf(pdf_path, max_pages):
    """Read raw PDF text via PyMuPDF."""
    if not Path(pdf_path).exists():
        raise FileNotFoundError(f"PDF not found: {pdf_path}")
    doc = fitz.open(pdf_path)
    text = "".join([page.get_text() for page in doc[:max_pages]])
    return text


def _strip_markdown_fences(text):
    """Remove markdown code fences (```text ... ```) from LLM output."""
    lines = text.split("\n")
    if lines and lines[0].strip().startswith("```"):
        lines = lines[1:]
    if lines and lines[-1].strip() == "```":
        lines = lines[:-1]
    return "\n".join(lines).strip()


def _normalize_whitespace(text):
    """Collapse single newlines to spaces, normalize quotes, preserve paragraph breaks."""
    # Normalize smart quotes to ASCII
    text = text.replace('\u2018', "'").replace('\u2019', "'")  # curly single quotes
    text = text.replace('\u201c', '"').replace('\u201d', '"')  # curly double quotes
    # Replace single newlines (not preceded/followed by another newline) with space
    text = re.sub(r'(?<!\n)\n(?!\n)', ' ', text)
    # Collapse multiple spaces
    text = re.sub(r' +', ' ', text)
    return text.strip()


def clean_policy_text(raw_text, disease_name, model_id):
    """Call Gemini to isolate disease section, remove PDF artifacts, preserve numbering."""
    print(f"\n🧹 Cleaning policy text for {disease_name}...")
    model = genai.GenerativeModel(model_id)
    prompt = textwrap.dedent(f"""
        Clean this raw PDF text for extraction.
        
        TASKS:
        1. Isolate the \"{disease_name}\" section ONLY.
           - Start at \"Initial Authorization\" or the disease name
           - Stop before the next disease or section
        2. Remove all footers, page numbers, headers, and copyrights.
        3. MERGE sentences broken by page breaks.
        4. PRESERVE the numbering structure ((1), (2), (a), (b), i, ii...) 
           exactly as written — it is vital for logic extraction.
        5. Output ONLY the cleaned plain text. Do NOT wrap in markdown code fences.
        
        RAW TEXT:
        {raw_text}
    """)
    try:
        response = model.generate_content(prompt)
        cleaned = response.text.strip()
        if not cleaned:
            raise ValueError(f"Cleaning returned empty text for {disease_name}")
        # Strip markdown code fences if Gemini wraps output
        cleaned = _strip_markdown_fences(cleaned)
        return cleaned
    except Exception as e:
        raise RuntimeError(f"Cleaning failed: {e}")


def extract_criteria(clean_text, disease_name, drug_name, auth_type, model_id):
    """
    Call LangExtract to extract PA criteria, logic nodes, and evidence requirements.

    Three extraction classes:
    - Criterion: Individual criteria and drug names (leaves)
    - LogicGate: Explicit logical operators (All of/One of/Both of the following)
    - EvidenceRequirement: Documentation requirements (what must be provided to justify a criterion)

    Returns: lx.Result object with flat extractions.
    """
    print(f"\n🔍 Extracting criteria + logic gates + evidence requirements for {drug_name} ({disease_name}) - {auth_type}...")

    prompt = textwrap.dedent(f"""
        Extract Prior Authorization approval criteria from a medical policy document.

        FOCUS: Extract ONLY "{auth_type}" criteria. Ignore Reauthorization or other sections.

        EXTRACTION CLASSES:
        - "LogicGate": Phrases defining logical structure ("all of the following", "one of the following", "both of the following").
        - "Criterion": Individual requirements, sub-requirements, or drug names.
        - "EvidenceRequirement": Documentation requirements that specify what must be submitted to justify a criterion.
          These are phrases like "document x, x, and x..." or "as documented by ...".

        ATTRIBUTES (required for each extraction):
        - type: One of "LogicGate", "Mandatory", "SubRequirement", "Drug", or "EvidenceRequirement"
        - logic_path: JSON array of strings representing the path in the decision tree.
          Use (AND) or (OR) suffixes to indicate logic type.
          For EvidenceRequirement, include the specific criterion text as the last path segment
          when the requirement applies to a specific criterion rather than the whole group.

        RULES:
        1. extraction_text MUST be a single string copied verbatim from the source text. NEVER return a list or array.
        2. When a criterion lists drugs in brackets [e.g., drug1, drug2], extract EACH drug as a SEPARATE Criterion extraction with type="Drug".
        3. Root logic is always AND: "{auth_type} (AND)"
        4. "One of the following" = OR, "Both of the following" = AND, "All of the following" = AND
        5. For negated criteria (e.g., "not receiving"), set negated="true" in attributes.
        6. Extract EvidenceRequirement entries for any criterion that specifies what documentation is needed.
           The logic_path should point to the specific criterion the requirement applies to.

        Policy: {disease_name} - {auth_type}
        Drug: {drug_name}
    """)

    # Use generic example text (not the actual policy) to avoid biasing the model
    examples = [
        lx.data.ExampleData(
            text=GENERIC_EXAMPLE_TEXT,
            extractions=GENERIC_EXAMPLE_EXTRACTIONS,
        )
    ]

    # Normalize whitespace in the input text
    normalized_text = _normalize_whitespace(clean_text)

    max_retries = 3
    last_error = None
    for attempt in range(max_retries):
        try:
            result = lx.extract(
                text_or_documents=normalized_text,
                prompt_description=prompt,
                examples=examples,
                model_id=model_id,
                extraction_passes=1,
                max_char_buffer=15000,
                max_workers=4,
                use_schema_constraints=False,
            )
            if not result.extractions:
                raise ValueError(f"No extractions returned for {disease_name} - {auth_type}")
            return result
        except (ValueError, RuntimeError) as e:
            last_error = e
            print(f"   Attempt {attempt + 1}/{max_retries} failed: {e}")
            if attempt < max_retries - 1:
                print(f"   Retrying in 2s...")
                time.sleep(2)

    raise RuntimeError(f"Extraction failed after {max_retries} attempts: {last_error}")


def _clean_segment_name(segment):
    """Remove (AND)/(OR) markers from segment name."""
    return segment.replace("(AND)", "").replace("(OR)", "").strip()


def _get_node_type(segment):
    """Extract logic type (AND/OR) from segment name."""
    return "OR" if "(OR)" in segment else "AND"


def _get_char_interval(item):
    """Extract char_interval dict from a LangExtract extraction item."""
    ci = getattr(item, "char_interval", None)
    if ci is None:
        return {}
    # Handle both dict and object with start_pos/end_pos attributes
    if isinstance(ci, dict):
        return ci
    return {"start_pos": getattr(ci, "start_pos", None), "end_pos": getattr(ci, "end_pos", None)}


def reconstruct_tree(extractions, disease_name):
    """
    Build nested tree from flat logic_path extractions.
    
    Preserves source_text and char_interval at ALL levels:
    - LogicGate extractions enrich parent nodes with the logic operator definition
    - EvidenceRequirement extractions are attached as evidence_requirements on their target node
      (can target leaf nodes when the logic_path includes the criterion text as the last segment)
    - Criterion/Drug extractions become leaf nodes
    """
    tree_root = {"name": f"{disease_name} Policy", "type": "ROOT", "children": []}

    if not extractions:
        return tree_root

    for item in extractions:
        path = item.attributes.get("logic_path", ["Uncategorized"])
        # Parse logic_path if it's a JSON string
        if isinstance(path, str):
            try:
                path = json.loads(path)
            except:
                path = [path]
        item_type = item.attributes.get("type", "Criterion")

        if isinstance(path, str):
            path = [path]

        # Navigate/build tree structure
        current_level = tree_root["children"]
        last_node = None

        for segment in path:
            clean_name = _clean_segment_name(segment)
            node_type = _get_node_type(segment)

            # Look for matching node by name first, then by source_text (for leaf nodes)
            found_node = next((n for n in current_level if n.get("name") == clean_name), None)
            if not found_node:
                found_node = next(
                    (n for n in current_level if n.get("source_text", "").startswith(clean_name)),
                    None
                )

            if not found_node:
                new_node = {"name": clean_name, "type": node_type, "children": []}
                current_level.append(new_node)
                found_node = new_node

            last_node = found_node
            current_level = found_node.get("children", [])

        if item_type == "EvidenceRequirement" and last_node:
            # Attach evidence requirement to the target node
            if "evidence_requirements" not in last_node:
                last_node["evidence_requirements"] = []
            req = {
                "source_text": item.extraction_text,
                "char_interval": _get_char_interval(item),
            }
            last_node["evidence_requirements"].append(req)
        elif item_type == "LogicGate" and last_node:
            # Enrich parent node with the logic gate definition from source
            if "source_text" not in last_node:
                last_node["source_text"] = item.extraction_text
            if "char_interval" not in last_node:
                last_node["char_interval"] = _get_char_interval(item)
        else:
            # Create leaf node for Criterion, Drug, SubRequirement, etc.
            negated_attr = item.attributes.get("negated", "false")
            negated = negated_attr.lower() == "true" if isinstance(negated_attr, str) else bool(negated_attr)
            leaf = {
                "type": "LEAF",
                "source_text": item.extraction_text,
                "char_interval": _get_char_interval(item),
                "negated": negated
            }
            current_level.append(leaf)

    return tree_root


def print_tree(node, indent="", is_last=True):
    """Pretty-print tree to console with ASCII art. Shows source_text, IDs, and evidence requirements."""
    if not isinstance(node, dict):
        return

    marker = "└── " if is_last else "├── "
    node_type = node.get("type", "LEAF")
    name = node.get("name", "Unnamed")

    if node_type == "LEAF":
        leaf_id = node.get("id", "")
        id_str = f" [{leaf_id}]" if leaf_id else ""
        print(f"{indent}{marker}📄 {name[:80]}{id_str}")
        source = node.get("source_text", "")
        if source:
            preview = source[:100].replace('\n', ' ')
            print(f"{indent}    └─ Source: {preview}...")
    else:
        negated = " [NEGATED]" if node.get("negated") else ""
        node_id = node.get("id", "")
        id_str = f" [{node_id}]" if node_id else ""
        source = node.get("source_text", "")
        source_str = ""
        if source:
            source_str = f"\n{indent}    └─ Source: {source[:100]}..."
        print(f"{indent}{marker}{node_type}: {name}{id_str}{negated}{source_str}")

    # Print evidence requirements if present
    ev_reqs = node.get("evidence_requirements", [])
    for req in ev_reqs:
        src_text = req.get("source_text", "")[:80]
        parts = [f'📋 Evidence: "{src_text}"']
        ev_indent = indent + ("    " if is_last else "│   ")
        print(f"{ev_indent}    ⚡ {' | '.join(parts)}")

    children = node.get("children", [])
    if isinstance(children, list):
        for i, child in enumerate(children):
            new_indent = indent + ("    " if is_last else "│   ")
            print_tree(child, new_indent, i == len(children) - 1)

/Users/tlavisse/Documents/code/medgemma-impact-tibotimz/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/h4/tgymlfrj5gd1vyw9f795rv8c0000gn/T/ipykernel_35131/4293852321.py:9: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [4]:
# ============================================================================
# CELL 3: RUN PIPELINE
# ============================================================================

try:
    # Step 1: Read PDF
    print(f"\n📄 Reading PDF: {POLICY_PDF}...")
    raw_text = read_pdf(POLICY_PDF, MAX_PDF_PAGES)
    print(f"   ✓ Loaded {len(raw_text)} characters")
    
    # Step 2: Clean text
    clean_text = clean_policy_text(raw_text, DISEASE_NAME, CLEANING_MODEL)
    print(f"   ✓ Cleaned to {len(clean_text)} characters")
    
    # Save clean text
    with open(OUTPUT_CLEAN_TEXT, "w", encoding="utf-8") as f:
        f.write(clean_text)
    print(f"   ✓ Saved clean text: {OUTPUT_CLEAN_TEXT}")
    
    # Step 3: Extract criteria with LangExtract
    result = extract_criteria(clean_text, DISEASE_NAME, DRUG_NAME, AUTH_TYPE, EXTRACTION_MODEL)
    print(f"   ✓ Extracted {len(result.extractions)} items for '{AUTH_TYPE}'")
    
    # Step 4: Save flat JSONL (for highlighting in downstream tools)
    lx.io.save_annotated_documents([result], output_name=OUTPUT_FLAT_JSONL, output_dir=OUTPUT_DIR)
    print(f"   ✓ Saved flat extractions: {OUTPUT_FLAT_JSONL}")
    
    # Step 5: Reconstruct, enrich, and save decision tree JSON
    tree = reconstruct_tree(result.extractions, f"{DISEASE_NAME} - {AUTH_TYPE}")
    tree = enrich_tree(tree)
    with open(OUTPUT_TREE_JSON, "w", encoding="utf-8") as f:
        json.dump(tree, f, indent=2, ensure_ascii=False)
    print(f"   ✓ Saved enriched decision tree: {OUTPUT_TREE_JSON}")
    
    # Step 6: Generate and save HTML visualization
    html_content = lx.visualize(OUTPUT_FLAT_JSONL)
    with open(OUTPUT_HTML, "w", encoding="utf-8") as f:
        if hasattr(html_content, 'data'):
            f.write(html_content.data)
        else:
            f.write(str(html_content))
    print(f"   ✓ Saved HTML visualization: {OUTPUT_HTML}")
    
    # Step 7: Pretty-print tree in notebook
    print(f"\n{'='*80}")
    print(f"DECISION TREE: {DISEASE_NAME} / {DRUG_NAME} / {AUTH_TYPE}")
    print(f"{'='*80}")
    print_tree(tree)
    print(f"\n✅ Pipeline complete! All outputs saved.")
    
except Exception as e:
    print(f"\n❌ Pipeline failed: {e}")
    import traceback
    traceback.print_exc()


📄 Reading PDF: UHC_Commercial_Medical_Policy_Adalimumab.pdf...
   ✓ Loaded 38615 characters

🧹 Cleaning policy text for Rheumatoid Arthritis...
   ✓ Cleaned to 2755 characters
   ✓ Saved clean text: rheumatoid_arthritis_initial_auth_clean.txt

🔍 Extracting criteria + logic gates + evidence requirements for Adalimumab (Rheumatoid Arthritis) - Initial Auth...


LangExtract: model=gemini-2.5-flash, current=2,755 chars, processed=0 chars:  [01:03]


   ✓ Extracted 31 items for 'Initial Auth'


LangExtract: Saving to rheumatoid_arthritis_initial_auth_extractions.jsonl: 1 docs [00:00, 839.70 docs/s]

✓ Saved 1 documents to rheumatoid_arthritis_initial_auth_extractions.jsonl


   ✓ Saved flat extractions: rheumatoid_arthritis_initial_auth_extractions.jsonl
   ✓ Saved enriched decision tree: rheumatoid_arthritis_initial_auth_decision_tree.json


LangExtract: Loading rheumatoid_arthritis_initial_auth_extractions.jsonl: 100%|██████████| 15.6k/15.6k [00:00<00:00, 46.5MB/s]

✓ Loaded 1 documents from rheumatoid_arthritis_initial_auth_extractions.jsonl
   ✓ Saved HTML visualization: rheumatoid_arthritis_initial_auth_visualization.html

DECISION TREE: Rheumatoid Arthritis / Adalimumab / Initial Auth
└── ROOT: Rheumatoid Arthritis - Initial Auth Policy
    └── AND: Initial Authorization [initial_authorization]
        └─ Source: all of the following criteria...
        ├── AND: Diagnosis [initial_authorization.diagnosis]
        │   └── 📄 Unnamed [initial_authorization.diagnosis.1_diagnosis_of_moderately_to_severely_active_rheumatoid_arth]
        │       └─ Source: (1) Diagnosis of moderately to severely active rheumatoid arthritis...
        ├── OR: Step Therapy [initial_authorization.step_therapy]
            └─ Source: One of the following:...
        │   ├── OR: Conventional DMARD Failure [initial_authorization.step_therapy.conventional_dmard_failure]
        │   │   ├── 📄 Unnamed [initial_authorization.step_therapy.conventional_dmard_failure.a_history_o

In [5]:
# ============================================================================
# CELL 4: DEMO — evaluate a patient against the tree
# ============================================================================

from policy_tree import load_tree, get_all_criteria, get_status, set_criterion, evaluate

# Step 1: Load the tree as PolicyNode objects (not raw dicts)
tree_node = load_tree("rheumatoid_arthritis_initial_auth_decision_tree.json")

# Step 2: See what criteria need to be evaluated
all_criteria = get_all_criteria(tree_node)
print(f"Found {len(all_criteria)} leaf criteria to evaluate:\n")
for cid, leaf in all_criteria.items():
    print(f"  {cid}")
    print(f"    → {leaf.source_text[:80]}")
    print()

# Step 3: Populate results for a specific patient
results = {}

set_criterion(results, 
    "initial_auth.diagnosis.1_diagnosis_of_moderately_to_severely_active_rheumatoid_arth",
    met=True,
    evidence="Patient diagnosed with moderate RA, DAS28 score 5.4",
    source_ref="Note_patient123_TARGET_diagnosis.txt")

set_criterion(results,
    "initial_auth.step_therapy.dmard_failure.methotrexate",
    met=True,
    evidence="3-month trial of methotrexate 25mg, stopped due to lack of efficacy",
    source_ref="Note_patient123_TARGET_treatment_failure.txt")

set_criterion(results,
    "initial_auth.combination_therapy.enbrel_etanercept",
    met=False)

set_criterion(results,
    "initial_auth.prescriber.4_prescribed_by_or_in_consultation_with_a_rheumatologist",
    met=True,
    evidence="Prescribed by Dr. Smith, Rheumatology",
    source_ref="Note_patient123_TARGET_prescriber.txt")

# Step 4: Evaluate
status = get_status(tree_node, results)
print(f"\nOverall: {status.overall}")
print(f"Met: {status.met_count}/{status.total_count}, Pending: {status.pending_count}")
for c in status.criteria:
    print(f"  [{c['status']}] {c['id']}")

Found 23 leaf criteria to evaluate:

  initial_auth.diagnosis.1_diagnosis_of_moderately_to_severely_active_rheumatoid_arth
    → (1) Diagnosis of moderately to severely active rheumatoid arthritis

  initial_auth.step_therapy.dmard_failure.methotrexate
    → methotrexate

  initial_auth.step_therapy.dmard_failure.leflunomide
    → leflunomide

  initial_auth.step_therapy.dmard_failure.sulfasalazine
    → sulfasalazine

  initial_auth.step_therapy.dmard_failure.hydroxychloroquine
    → hydroxychloroquine

  initial_auth.step_therapy.prior_biologic.enbrel_etanercept
    → Enbrel (etanercept)

  initial_auth.step_therapy.prior_biologic.cimzia_certolizumab
    → Cimzia (certolizumab)

  initial_auth.step_therapy.prior_biologic.simponi_golimumab
    → Simponi (golimumab)

  initial_auth.step_therapy.prior_biologic.orencia_abatacept
    → Orencia (abatacept)

  initial_auth.step_therapy.prior_biologic.xeljanz_tofacitinib
    → Xeljanz (tofacitinib)

  initial_auth.step_therapy.prior_biologic